In [ ]:
# !pip install -r requirements.txt

In [ ]:
# Mounting the drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Shortcut to our folder
import os
os.chdir("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab")

In [ ]:
import pandas as pd
import geopandas as gpd

### Import departure times and lodes data.

Departure times contains estimate counts of departure times per tract

Lodes data contains home and work tracts

In [ ]:
departure_times_cleaned = pd.read_csv("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab/Data/acs5_2023_Bay_Area_departure_times_cleaned.csv")

departure_times_cleaned.head(5)

In [ ]:
# read cleaned tract populations data
tract_pop = pd.read_parquet("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab/Data/tract_pop.parquet")

### Plot EPC Communities

In [ ]:
import geopandas as gpd

In [ ]:
# Here is the section for loading and cleaning EPC Data
epc = gpd.read_file("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab/Data/MTC EPC data/EPC_2020_acs2018.geojson")

In [ ]:
epc.head(2)

In [ ]:
epc.columns

In [ ]:
epc.plot()

In [ ]:
#combine with regular map
epc.crs

In [ ]:
#loading in the cleaned Bay Counties parquet file
bay_counties = gpd.read_parquet("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab/Data/bay_counties_cleaned.parquet")

In [ ]:
#sanity check plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 10))

bay_counties.plot(ax = ax, color = "lightgrey", edgecolor = "grey")
epc.plot(ax = ax)

plt.show()

In [ ]:
print(len(bay_counties))
#bay_counties

In [ ]:
#NEW CELL FROM WYLIE - MERGING EPC & BAY COUNTIES

#sanity check that we should use the epc_2050 column to define EPCs
print(f'Total number of EPCs: {len(epc)}')

print(f'Number of 2050 EPCs: {epc[['epc_2050']].sum()}')
print(f'Number of 2040 EPCs: {epc[['epc_2040']].sum()}')
print(f'Number of 2035 EPCs: {epc[['epc_2035']].sum()}')

In [ ]:
#NEW CELL FROM WYLIE - MERGING EPC & BAY COUNTIES

#merging bay_counties and epc data

epc_relevant_columns = ['geoid', 'epc_2050']

bay_counties = bay_counties.merge(
    epc[epc_relevant_columns], #only merge the relevant columns
    left_on="GEOID",
    right_on="geoid",
    how="left"
)

In [ ]:
#NEW CELL FROM WYLIE - MERGING EPC & BAY COUNTIES

#adding a boolean column, True = tract is EPC, False = tract is not EPC
bay_counties["epc_2050"] = bay_counties["epc_2050"].notna()

#rename to make it clear it's a bool
bay_counties = bay_counties.rename(columns={"epc_2050": "is_epc_2050"})

bay_counties.head(3)

In [ ]:
#saving the merged gdf to our drive, called bay_tracts_with_epcs.parquet
# ^ if you can think of a clearer name than this, pls feel free to change lol

bay_counties.to_parquet("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab/Data/bay_tracts_with_epcs.parquet")


### Plotting Daytime Population

In [ ]:
#merge on GEOID and work_tract columns, for daytime populations.
bay_counties = pd.merge(
    bay_counties, tract_pop,
    how = "left",
    left_on = "GEOID",
    right_index = True
    )
bay_counties.head(2)

In [ ]:
#NEW CELL FROM WYLIE, CHECKING THE DTYPES OF DEPARTURE TIMES

departure_times_cleaned.info()
#okay the datatypes are all wack from loading in the file? Let's clean them again

In [ ]:
#NEW CELL FROM WYLIE, FIXING THE DTYPES OF DEPARTURE TIMES
cols_to_str = ['state','county','tract','GEOID']

departure_times_cleaned[cols_to_str] = departure_times_cleaned[cols_to_str].astype(str)

#original integer dtype got rid of the leading zero on GEOID, add back in:
departure_times_cleaned["GEOID"] = departure_times_cleaned["GEOID"].str.zfill(11)

departure_times_cleaned["GEOID"].head(3)

In [ ]:
#add a new column called Base_Pop: match GEOID, use total commuter pop from departure_times_cleaned
#TODO: need to look at home_pop vs total_pop (from ACS), numbers aren't as expected
bay_counties = pd.merge(
    bay_counties, departure_times_cleaned[["GEOID", "total_pop","total_commuters"]],
    how = "left",
    on = "GEOID"
)

bay_counties.head()

In [ ]:
#rename columns and add total daytime population

bay_counties = bay_counties.rename(columns = {"total_commuters": "resident_pop"})
# bay_counties["Pop_Total_Daytime"] = bay_counties["Pop_Change_Daytime"] + bay_counties["Pop_Resident"]
bay_counties.head(3)
print(bay_counties.columns.to_list())

In [ ]:
# is_na = bay_counties[bay_counties["Pop_Total_Daytime"].isna()]

## Plotting the percentage of tract total population who are commuters

In [ ]:
#making a list of the columns we don't need
cols_to_drop = ["STATEFP", "COUNTYFP", "TRACTCE", "NAME", "NAMELSAD", "MTFCC", 'ALAND', 'AWATER', 'geoid', 'work_pop', 'home_pop', 'within_tract']

#make a copy of gdf with only necessary cols
commuter_pcts = bay_counties.drop(columns = cols_to_drop)
commuter_pcts.head(3)

In [ ]:
#calculating the percentage of total population

commuter_pcts['commuter_pct'] = 100* commuter_pcts['resident_pop'] / commuter_pcts['total_pop']

commuter_pcts['commuter_pct'].head(3)

In [ ]:
# #plotting the percentage of commuters in each tract

# commuter_pcts.plot(
#     column = "commuter_pct",
#     legend = True,
#     legend_kwds={
#         'label': "Commuter percentage",
#         'orientation': "vertical",
#         'shrink': 0.8
#     }
#     )

# plt.title("Percentage of total population who are commuters in the Bay Area")
# plt.show()

In [ ]:
import folium

# custom bins to have round values in legend
custom_bins = [10, 20, 25, 30, 35, 40, 45, 50, 60]
m = commuter_pcts.explore(
    column="commuter_pct",
    scheme="UserDefined",
    classification_kwds={"bins": custom_bins},
    cmap="Greens", #green hues (to be different from day & nighttime colours)
    legend=True,
    legend_kwds={"caption": "Commuter Percentage"},
    tiles="CartoDB positron",
    tooltip=["GEOID", "commuter_pct", "is_epc_2050"],
    popup=["GEOID", "commuter_pct", "is_epc_2050"],
    style_kwds={
        "weight": 0.3,
        "fillOpacity": 0.7
    },
    missing_kwds={
        "color": "#a8d5e2", #blue to look like water
        "fillOpacity": 0.8
    },
    name="Commuter Percentage"
)

commuter_pcts[commuter_pcts["is_epc_2050"]].explore(
    m=m,
    color="none",
    style_kwds={
        "color": "#FF0090",   #pink contrasts against green, colorblind safe
        "weight": 1,
        "fillOpacity": 0,
        "fill": False
    },
    tooltip=False,
    name="EPC Tracts (2050)"
)

folium.LayerControl().add_to(m)

m

## Plotting the daytime and resident commuter distributions, and their net change

In [ ]:
bay_counties_plot = bay_counties.drop(columns = ["STATEFP", "COUNTYFP", "TRACTCE", "NAME", "NAMELSAD", "MTFCC", 'ALAND', 'AWATER'])
bay_counties_plot["daytime_pop"] = bay_counties_plot["resident_pop"] + bay_counties_plot["work_pop"] - bay_counties_plot["within_tract"]
bay_counties_plot.head(3)

In [ ]:
print(bay_counties_plot['work_pop'].min())
print(bay_counties_plot['work_pop'].max())
print(bay_counties_plot['home_pop'].min())
print(bay_counties_plot['home_pop'].max())

In [ ]:
#saving the final gdf as a parquet

bay_counties_plot.to_parquet("/content/drive/MyDrive/Urban_Informatics_Final_Project/Colab/Data/day_night_pop_change.parquet")

In [ ]:
#Non-interactive map code

# bay_counties_plot.plot(
#     column = "daytime_pop",
#     legend = True,
#     legend_kwds={
#         'label': "Daytime Population",
#         'orientation': "vertical",
#         'shrink': 0.8
#     },
#     vmax = 50000
#     )

# plt.title("Daytime Population in 9 Bay Area Counties")
# plt.show()

In [ ]:
# #TODO: Check numbers because these plots look funny

# #Non-interactive map code

# bay_counties_plot.plot(
#     column = "resident_pop",
#     legend = True,
#     legend_kwds={
#         'label': "Resident in Tract",
#         'orientation': "vertical",
#         'shrink': 0.8
#     },
#     vmax = 10000
#     )

# plt.title("Resident Population in 9 Bay Area Counties")
# plt.show()

In [ ]:
#NEW CELL FROM WYLIE - INTERACTIVE VIS

#This line is basically changing any areas with resident pop = 0 to resident pop = NaN
#so that the .explore() functionality can black out the water areas based on the
#condition resident pop = NaN
bay_counties_plot.loc[bay_counties_plot["resident_pop"] == 0, "resident_pop"] = None

import folium

# custom bins to have round values in legend
custom_bins = [5000, 10000, 15000, 20000, 30000, 40000, 50000]

#Base colormap of daytime pop
m = bay_counties_plot.explore(
    column="daytime_pop",
    scheme="UserDefined",
    classification_kwds={"bins": custom_bins},
    cmap="Oranges", #Hue colormap, evokes thoughts of 'daytime', colorblind accessible
    legend=True,
    legend_kwds={"caption": "Daytime Population"},
    vmax=50000,
    tiles="CartoDB positron",
    tooltip=["GEOID", "daytime_pop", "is_epc_2050", "work_pop", "resident_pop"],
    popup=["GEOID", "daytime_pop", "is_epc_2050", "work_pop", "resident_pop"],
    style_kwds={
        "weight": 0.3,        #thin borders for non-EPC tracts
        "fillOpacity": 0.7    #some seethrough-ness to see basemap underneath
    },
    missing_kwds={
        "color": "#a8d5e2",       #light blue to look like water
        "fillOpacity": 0.8
        },
    name="Daytime Population"
)

#EPC overlay to outline them
bay_counties_plot[bay_counties_plot["is_epc_2050"]].explore(
    m=m,
    color="none",
    style_kwds={
        "color": "blue",      #blue outline to contrast, colorblind accessible
        "weight": 1,          #thicker border for EPCs
        "fillOpacity": 0,
        "fill": False
    },
    tooltip=False,
    name="EPC Tracts (2050)"
)

#Layer control to toggle EPC on/off
folium.LayerControl().add_to(m)

m

In [ ]:
#checking what the 90th percentile is for vmax of resident population
bay_counties_plot["resident_pop"].quantile(0.90)

In [ ]:
#WYLIE CELL - SAME AGAIN FOR NIGHT POP
#TODO: check the values, they seem low
import folium

# custom bins to have round values in legend
custom_bins = [500, 1000, 1500, 2000, 2500, 3000,]

#Base colormap of resident pop
m = bay_counties_plot.explore(
    column="resident_pop",
    scheme="UserDefined",
    classification_kwds={"bins": custom_bins},
    cmap="Purples", #Hue colormap, evokes thoughts of 'night', colorblind accessible
    legend=True,
    legend_kwds={"caption": "Nighttime Population","fmt": "{:.0f}"},
    vmax=2630, #setting max = 90th percentile
    tiles="CartoDB positron",
    tooltip=["GEOID", "daytime_pop", "is_epc_2050", "work_pop", "resident_pop"],
    popup=["GEOID", "daytime_pop", "is_epc_2050", "work_pop", "resident_pop"],
    style_kwds={
        "weight": 0.3,            #thin borders for non-EPC tracts
        "fillOpacity": 0.7,
        },
    missing_kwds={
        "color": "#a8d5e2",       #light blue to look like water
        "fillOpacity": 0.8
        },
    name="Nighttime Population"
)

#EPC overlay to outline them
bay_counties_plot[bay_counties_plot["is_epc_2050"]].explore(
    m=m,
    color="none",
    style_kwds={
        "color": "#FFB700",        #yellow outline to contrast, colorblind accessible
        "weight": 1.5,              #thicker border for EPCs
        "fillOpacity": 0,
        "fill": False
    },
    tooltip=False,
    name="EPC Tracts (2050)"
)

#Layer control to toggle EPC on/off
folium.LayerControl().add_to(m)

m

## Net change in population (histograms and map)

In [ ]:
#WYLIE CELL - GETTING THE NET CHANGE OF DAYTIME POP

bay_counties_plot["net_worker_change"] = bay_counties_plot["work_pop"] - bay_counties_plot["resident_pop"]

In [ ]:
#we want the diverging colormap to be white at zero, so need to have equal min & max
#will set this to the 95th percentile again

bay_counties_plot["net_worker_change"].quantile(0.05)


In [ ]:
#PLOTTING NET CHANGE

# custom bins to have round values in legend
custom_bins = [-5000, -4000, -3000, -2000, -1000, 0, 1000, 2000, 3000, 4000, 5000]


# Base colormap of net change
m = bay_counties_plot.explore(
    column="net_worker_change",
    scheme="UserDefined",
    classification_kwds={"bins": custom_bins},
    cmap="PuOr_r", #Diverging colormap for negative & positive, colorblind accessible
    legend=True,
    legend_kwds={"caption": "Net Daytime Worker Change","fmt": "{:.0f}"},
    vmax=6000, #setting max to be around 95th percentile
    vmin=-6000, #has to be = -vmax so that 0 is white
    tiles="CartoDB positron",
    tooltip=["GEOID", "net_worker_change", "is_epc_2050"],
    popup=["GEOID", "net_worker_change", "is_epc_2050"],
    style_kwds={
        "weight": 0.3,            # thin borders for non-EPC tracts
        "fillOpacity": 0.7,
        },
    missing_kwds={                #make the water blue like the daytime map
        "color": "#a8d5e2",
        "fillOpacity": 0.8
        },
    name="Net Daytime Worker Change"
)

#EPC overlay to outline them
bay_counties_plot[bay_counties_plot["is_epc_2050"]].explore(
    m=m,
    color="none",
    style_kwds={
        "color": "blue",       #blue outline to contrast, colorblind accessible
        "weight": 1,           #thicker border for EPCs
        "fillOpacity": 0,
        "fill": False
    },
    tooltip=False,
    name="EPC Tracts (2050)"
)

#Layer control to toggle EPC on/off
folium.LayerControl().add_to(m)

m

### Histograms

##### Wylie's sanity check histogram

In [ ]:
# #data seems skewed positive, so we're not really seeing the negative change in color
# #check the skew

# bay_counties_plot["net_worker_change"].describe()
# bay_counties_plot["net_worker_change"].hist(bins=50)

##### Quinten's histograms

In [ ]:
# histogram of net change in daily pop at tract level
import seaborn as sns

ax = sns.histplot(bay_counties_plot,
             x='net_worker_change',
             bins=500,
             kde = True
             )

ax.set(xlabel="Net change in daily population", ylabel="Tract Count",
       title="Net Daily Population Change Distribution for All Tracts");
# semicolon mutes text outputs

plt.savefig("Visuals/netpopchange_alltracts.png", dpi=300, bbox_inches="tight")

In [ ]:
# edited bounds to zoom in on main hump

ax = sns.histplot(bay_counties_plot,
             x='net_worker_change',
             bins=500,
             kde=True   # we can remove this if we think its not valuable
             )

ax.set(xlabel="Net change in daily population", ylabel="Tract Count",
       title="Net Daily Population Change Distribution for All Tracts");
# semicolon mutes text outputs

ax.set_xlim(-5000, 10000);
# setting manual x axis limits to not see outliers and get better idea of actual
# distribution shape

plt.savefig("Visuals/netpopchange_alltracts_cropped.png", dpi=300, bbox_inches="tight")

In [ ]:
#plot another that is specifically EPC tracts

ax = sns.histplot(bay_counties_plot[bay_counties_plot["is_epc_2050"]],
             x='net_worker_change',
             bins=500,
             kde=True,   # we can remove this if we think its not valuable
             color = "orange"
             )

ax.set(xlabel="Net change in daily population", ylabel="Tract Count",
       title="Net Daily Population Change Distribution for EPCs");
# semicolon mutes text outputs

# setting manual x axis limits to not see outliers and get better idea of actual
# distribution shape

plt.savefig("Visuals/netpopchange_EPCtracts.png", dpi=300, bbox_inches="tight")

In [ ]:
#plot another that is specifically EPC tracts

ax = sns.histplot(bay_counties_plot[bay_counties_plot["is_epc_2050"]],
             x='net_worker_change',
             bins=500,
             kde=True, # we can remove this if we think its not valuable
             color = "orange"
             )

ax.set(xlabel="Net change in daily population", ylabel="Tract Count",
       title="Net Daily Population Change Distribution for EPCs");
# semicolon mutes text outputs

ax.set_xlim(-5000, 10000);
# setting manual x axis limits to not see outliers and get better idea of actual
# distribution shape

plt.savefig("Visuals/netpopchange_EPCtracts_cropped.png", dpi=300, bbox_inches="tight")


In [ ]:
# plot zoomed in versions together

# all tracts in blue
sns.histplot(bay_counties_plot,
             x='net_worker_change',
             bins=500,
             kde=True,
             color="skyblue",
             label="All Tracts")

# EPC tracts in orange
sns.histplot(bay_counties_plot[bay_counties_plot["is_epc_2050"]],
             x='net_worker_change',
             bins=500,
             kde=True,
             color="orange",
             label="EPCs 2050")

plt.xlim(-5000, 10000)
plt.xlabel("Net change in daily population")
plt.ylabel("Tract Count")
plt.title("Net Daily Population Change Distribution: All tracts vs. EPCs")
plt.legend() # This adds the labels to the corner

plt.savefig("Visuals/netpopchange_overlaid.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# plot zoomed in versions together

# all tracts in blue
sns.histplot(bay_counties_plot,
             x='net_worker_change',
             bins=500,
             kde=True,
             color="skyblue",
             label="All Tracts")

# EPC tracts in orange
sns.histplot(bay_counties_plot[bay_counties_plot["is_epc_2050"]],
             x='net_worker_change',
             bins=500,
             kde=True,
             color="orange",
             label="EPCs 2050")

plt.yscale('log')
plt.ylim(1,200)
plt.xlim(-5000, 10000)
plt.xlabel("Net change in daily population")
plt.ylabel("Tract Count")
plt.title("Net Daily Population Change Distribution: All tracts vs. EPCs")
plt.legend() # This adds the labels to the corner

plt.savefig("Visuals/netpopchange_overlaid_logscale.png", dpi=300, bbox_inches="tight")

plt.show()

Note for later:
1. Maybe another version of this weighted by tract population
2. Maybe another version of this normalized to compare the level of peaking in each histogram